# **RAG Ingestion Pipeline**

## **Imports and Configuration**

In [ ]:
# --- Notebook setup and autoreload configuration ---
# This cell runs the initial setup script and enables the autoreload extension.
# The autoreload feature ensures that updates made to imported modules (e.g., in src/)
# are automatically reloaded without restarting the kernel.
%run notebook_setup.py

%load_ext autoreload

%autoreload 2

In [ ]:
import logging

from backend.pipelines.etl.data_pipeline import DataPipeline
from backend.pipelines.chunking.chunking_pipeline import ChunkingPipeline
from backend.pipelines.vectorstore.vectorstore_pipeline import VectorStorePipeline
from backend.utils.logger import setup_logging

In [ ]:
logger = logging.getLogger("INGESTION_WORKFLOW")

RAW_FILENAME = "wiki_movie_plots_deduped.csv"
DOCS_JSONL_FILENAME = "docs.jsonl"
CHUNKS_JSONL_FILENAME = "chunks.jsonl"

if not logging.getLogger().handlers:
    setup_logging()

# NOTE: `project_root` is defined by `%run notebook_setup.py`
assert "project_root" in globals(), "Run: %run notebook_setup.py"

raw_path =  (project_root / "data" / "raw" / RAW_FILENAME).resolve()
processed_dir =  (project_root / "data" / "processed").resolve()
processed_dir.mkdir(parents=True, exist_ok=True)

## **ETL - Data Cleaning and Structured Output Generation**

This step runs the **ETL pipeline** responsible for preparing the raw dataset of movie plots for subsequent stages of the RAG system.

The process includes **three key operations**:
* `Data Cleaning`:
    * The **raw CSV** file is loaded and cleaned using the `DataCleaner` class (`data_cleaner`), which removes invalid tokens, standardizes formats, and applies column-specific transformations.
* `JSONL Creation`:
    * The `JsonlWriter` class (`jsonl_writer`) converts the structured dataset into **JSON Lines (.jsonl)** format, separating the main text (`Plot`) from its **metadata** (e.g., title, genre, year, cast, director).

In summary, this step transforms the raw movie dataset into a clean, structured, and RAG-ready format — enabling reliable chunking and semantic representation.


In [ ]:
logger.info("Starting ETL step")

jsonl_out_path = processed_dir / DOCS_JSONL_FILENAME
pipeline = DataPipeline(
    raw_path=raw_path,
    jsonl_out_path=jsonl_out_path
)
pipeline.run()

_Expected output_:
* `docs.jsonl` (RAG-ready JSON lines file)

## **Chunking - Text Splitting for Document Segmentation**

This step splits the cleaned movie plot texts into smaller, context-preserving chunks suitable for embedding and retrieval.

It performs **three major operations**:
* `Chunk Loading`:
    * The `.jsonl` file produced during ETL is loaded and parsed into a collection of documents containing `text` and `metadata`.
* `Splitting Strategy Application`:
    * The `ChunkingPipeline` applies the configured chunking strategy (character, token, or recursive), using parameters defined in `CHUNKING_CONFIG` to determine chunk size and overlap.
    * This ensures contextual continuity while limiting token counts per segment.
* `Chunk Output Serialization`:
    * Each document’s text is split into multiple sub-documents, preserving metadata such as `Title`, `Release Year`, and a unique `id`.
    * The resulting collection is stored in a new `.jsonl` file (`chunks.jsonl`) for embedding.

This stage ensures that the RAG system operates over optimized text units - large enough to preserve context but small enough for efficient vector search.

In [ ]:
logger.info("Starting Chunking step")

in_path = processed_dir / DOCS_JSONL_FILENAME
out_path = processed_dir / CHUNKS_JSONL_FILENAME
pipeline = ChunkingPipeline(input_path=in_path, output_path=out_path)
pipeline.run()

_Expected output_:
* `chunks.jsonl` file containing segmented text chunks with metadata.

## **VectorStore - Embedding and Persistence**

In this stage, the pre-chunked documents are transformed into vector representations and stored in a persistent vector database.

The process includes the following steps:
* `Chunk Loading`:
    * The `chunks.jsonl` file is read and converted into `langchain_core.documents.Document` objects, combining each text segment with its corresponding metadata.
* `Embedding Generation`:
    * The `VectorStorePipeline` uses `OpenAIEmbeddings`, configured with the `text-embedding-3-small` model (as defined in EMBEDDING_CONFIG), to generate dense semantic vectors for each chunk.
* `Vector Storage`:
    * These vectors are persisted into a local **ChromaDB** instance (`VECTORSTORE_CONFIG["persist_dir"]`), enabling fast similarity-based retrieval for the RAG retriever.

This step converts textual knowledge into an efficient, queryable vector representation — the foundation for semantic search and contextual question answering.

In [ ]:
logger.info("Starting VectorStore step")

in_path = processed_dir / CHUNKS_JSONL_FILENAME
pipeline = VectorStorePipeline(input_path=in_path)
pipeline.run()

## **Notebook Summary**

**This notebook allows you to**:
* Run and debug each ingestion step individually (ETL, Chunking, and VectorStore).
* Inspect intermediate artifacts (`docs.jsonl`, `chunks.jsonl`) and verify their structural and semantic integrity.
* Generate embeddings for pre-chunked documents and persist them into a local vector database (`ChromaDB`).
* Validate that the full ingestion pipeline, from raw CSV to a persistent vector store, produces retrieval-ready representations suitable for downstream RAG question-answering workflows.